# Bölüm 3 — LLM ile Metin Analizi

LLM: Groq üzerinden `openai/gpt-oss-20b`, `temperature=0`. Etiketlemede (Prompt 1 ve 2) modelin varsayılan akıl yürütme ayarı, 3.3'teki önerilerde `reasoning_effort='low'` kullanıldı.
(Gemini ücretsiz katmanı sürekli 503 "aşırı yoğunluk" hatası verdiği için Groq'a geçildi. Önce `gpt-oss-120b` kullanıldı; günlük token kotası dolduğu için aynı ailenin küçük modeline geçildi. İki etiketleme prompt'u da aynı modelle ve aynı ayarlarla çalıştırıldı.)

In [2]:
# --- Kurulum ---
import os
import re
import json
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore', category=RuntimeWarning)
VERI = '../veri/'
CIKTI = '../ciktilar/'
KESIM = pd.Timestamp('2025-06-30')

# LLM çağrıları yavaş ve her çalıştırmada küçük farklar verebilir.
# False: kayıtlı sonuç dosyaları kullanılır | True: yorumlar LLM'e yeniden gönderilir
LLM_YENIDEN = False

GECERLI_DUYGU = ['Olumlu', 'Olumsuz', 'Karışık', 'Alakasız']
GECERLI_KONU = ['randevu', 'fiyat', 'süre', 'personel', 'iş_kalitesi', 'yedek_parça', 'temizlik', 'diğer']

In [6]:
# Etiketleme kılavuzu: insan etiketleyici ve LLM (prompt 2) aynı kuralları kullanıyor
# Duygu etiketleri vakanın verdiği dört değer; her biri için tanım
KILAVUZ = {
    'genel': 'Tek tek kelimelere değil, metnin gerçek anlamına göre karar ver.',
    'duygu': {
        'Olumlu':   'Hizmet olumlu değerlendiriliyor, olumsuz bir değerlendirme yok.',
        'Olumsuz':  'Hizmet olumsuz değerlendiriliyor, olumlu bir değerlendirme yok.',
        'Karışık':  'Olumlu ve olumsuz yönler bir arada, ya da ne açıkça olumlu ne açıkça olumsuz olan çekingen bir değerlendirme ',
        'Alakasız': 'Hizmete dair hiçbir değerlendirme yok (anlamsız işaret, konu dışı metin). '
                    
    },
    'konular': {
        'K1': 'Yalnızca metinde açıkça değerlendirilen konuları etiketle. Metinde olmayanı çıkarma.',
        'K2': 'Somut bir konu yoksa (genel ifade, teşekkür, anlamsız metin) konular boş bırakılır.',
        'K3': 'diğer: Somut bir konu var ama listedeki hiçbirine uymuyor.',
    },
    'konu_tanimlari': {
        'randevu': 'randevu alma ve randevu süreci',
        'fiyat': 'ücret ve fatura',
        'süre': 'bekleme ve işin ne kadar sürdüğü',
        'personel': 'çalışanların tavrı ve iletişimi',
        'iş_kalitesi': 'yapılan onarım veya bakımın kalitesi',
        'yedek_parça': 'araca takılan parçalar',
        'temizlik': 'aracın veya servisin temizliği',
        'diğer': 'listedeki hiçbirine uymayan somut bir konu',
    },
}

def kilavuz_metni():
    """Kılavuzu okunabilir metne çevirir (dosyaya kaydetmek)."""
    satirlar = [f"Genel ilke: {KILAVUZ['genel']}", '', '## Duygu (tam olarak biri)']
    satirlar += [f'- {k}: {v}' for k, v in KILAVUZ['duygu'].items()]
    satirlar += ['', '## Konular (birden fazla olabilir; yoksa boş)']
    satirlar += [f'- {k}: {v}' for k, v in KILAVUZ['konular'].items()]
    satirlar += ['', '## Konu tanımları']
    satirlar += [f'- {k}: {v}' for k, v in KILAVUZ['konu_tanimlari'].items()]
    return '\n'.join(satirlar)

with open(CIKTI + 'etiketleme_kilavuzu.md', 'w', encoding='utf-8') as f:
    f.write('# Etiketleme kılavuzu\n\n' + kilavuz_metni())
print(kilavuz_metni())

Genel ilke: Tek tek kelimelere değil, metnin gerçek anlamına göre karar ver.

## Duygu (tam olarak biri)
- Olumlu: Hizmet olumlu değerlendiriliyor, olumsuz bir değerlendirme yok.
- Olumsuz: Hizmet olumsuz değerlendiriliyor, olumlu bir değerlendirme yok.
- Karışık: Olumlu ve olumsuz yönler bir arada, ya da ne açıkça olumlu ne açıkça olumsuz olan çekingen bir değerlendirme 
- Alakasız: Hizmete dair hiçbir değerlendirme yok (anlamsız işaret, konu dışı metin). 

## Konular (birden fazla olabilir; yoksa boş)
- K1: Yalnızca metinde açıkça değerlendirilen konuları etiketle. Metinde olmayanı çıkarma.
- K2: Somut bir konu yoksa (genel ifade, teşekkür, anlamsız metin) konular boş bırakılır.
- K3: diğer: Somut bir konu var ama listedeki hiçbirine uymuyor.

## Konu tanımları
- randevu: randevu alma ve randevu süreci
- fiyat: ücret ve fatura
- süre: bekleme ve işin ne kadar sürdüğü
- personel: çalışanların tavrı ve iletişimi
- iş_kalitesi: yapılan onarım veya bakımın kalitesi
- yedek_parça: araca t

## 3.1 Örneklem

In [3]:
# Amaç: Geri bildirimlere genel bakış
yorumlar = pd.read_csv(VERI + 'geri_bildirimler.csv')
print('Toplam yorum:', len(yorumlar), '| Boş metin:', yorumlar['metin'].isna().sum(),
      '| Farklı metin:', yorumlar['metin'].nunique())

# Boş metinler etiketlenemez → çıkarılıyor. Rastgele 200 yorum (random_state=42 → herkes aynı 200'ü alır)
dolu = yorumlar.dropna(subset=['metin'])
orneklem = dolu.sample(n=200, random_state=42)
orneklem.to_csv(CIKTI + 'orneklem_200.csv', index=False)
print('Seçilen yorum:', len(orneklem), '| İçindeki farklı metin:', orneklem['metin'].nunique())

Toplam yorum: 640 | Boş metin: 11 | Farklı metin: 35
Seçilen yorum: 200 | İçindeki farklı metin: 35


In [5]:
# Kör etiketleme şablonu (kılavuza göre)
# - 35 farklı metin karışık sırada. 
# - LLM cevapları bu dosyada yok
SABLON = CIKTI + 'altin_etiket.csv'
if not os.path.exists(SABLON):
    farkli = pd.DataFrame({'metin': orneklem['metin'].drop_duplicates().sample(frac=1, random_state=7).values})
    farkli['duygu'] = ''
    farkli['konular'] = ''
    farkli['kural'] = ''     # hangi kurala dayandığın (ör. D3, K1) — belirsiz durumlarda doldur
    farkli['not'] = ''
    farkli.to_csv(SABLON, index=False, sep=';', encoding='utf-8-sig')
    print('Şablon oluşturuldu:', len(farkli), 'metin →', SABLON)
else:
    print('altin_etiket.csv zaten var, üzerine yazılmadı')

Şablon oluşturuldu: 35 metin → ../ciktilar/altin_etiket.csv


## Altın set (elle etiketleme)
35 farklı metin, yukarıdaki kılavuza göre, LLM çıktısı görülmeden etiketlendi. 200 yorumun tamamı bu 35 metinden oluştuğu için, her yorumun altın etiketi metnine göre eşleştiriliyor.

In [7]:
# Elle etiketleri oku ve standartlaştır (Excel'den gelen biçim farkları: büyük/küçük harf, boşluk, ayırıcı)
altin = pd.read_csv(CIKTI + 'altin_etiket.csv', sep=None, engine='python', encoding='utf-8-sig').fillna('')
for s in altin.columns:
    altin[s] = altin[s].astype(str).str.strip()

altin['duygu'] = altin['duygu'].str.capitalize()

def konu_duzelt(k):
    """'Personel , iş kalitesi' → 'personel, iş_kalitesi' (virgülle ayır, boşlukları alt çizgiye çevir)"""
    parcalar = [re.sub(r'\s+', '_', p.strip()).lower() for p in k.split(',')]
    return ', '.join(p for p in parcalar if p)

altin['konular'] = altin['konular'].apply(konu_duzelt)

# Kontroller
def gecersiz_konular(k):
    return [x for x in k.split(', ') if x and x not in GECERLI_KONU]

print('Satır:', len(altin))
print('Boş duygu:', (altin['duygu'] == '').sum())
print('Geçersiz duygu:', (~altin['duygu'].isin(GECERLI_DUYGU)).sum())
print('Geçersiz konu:', altin['konular'].apply(gecersiz_konular).str.len().gt(0).sum())
print('Alakasız ama konusu dolu (K2 ihlali):', ((altin['duygu'] == 'Alakasız') & (altin['konular'] != '')).sum())
print('\nDuygu dağılımı:', altin['duygu'].value_counts().to_dict())

altin.to_csv(CIKTI + 'altin_etiket.csv', index=False, encoding='utf-8-sig')   # standartlaştırılmış hali

Satır: 35
Boş duygu: 0
Geçersiz duygu: 0
Geçersiz konu: 0
Alakasız ama konusu dolu (K2 ihlali): 0

Duygu dağılımı: {'Olumsuz': 14, 'Olumlu': 11, 'Karışık': 8, 'Alakasız': 2}


## 3.1 LLM ile etiketleme

In [ ]:
import getpass
from groq import Groq

# API anahtarı koda yazılmıyor (teslim edilen dosyada görünmesin diye); çalıştırırken sorulur
if 'GROQ_API_KEY' not in os.environ:
    os.environ['GROQ_API_KEY'] = getpass.getpass('Groq API anahtarı: ')
groq_client = Groq()
MODEL = 'openai/gpt-oss-20b'
REASONING = 'low' 

# Bağlantı testi (veride olmayan bir cümleyle)
cevap = groq_client.chat.completions.create(
    model=MODEL,
    messages=[{'role': 'user', 'content': 'Şu cümlenin duygusu olumlu mu olumsuz mu? Tek kelimeyle: "Kahve soğuktu."'}],
    temperature=0,
)
print(cevap.choices[0].message.content)

Olumsuz


In [20]:
# Prompt 1: yalnızca etiket listesi; kural ve tanım yok → LLM'in varsayılan davranışını ölçmek için
PROMPT_1 = """Aşağıdaki müşteri geri bildirimini etiketle.

duygu: Şu dört değerden tam olarak biri: Olumlu, Olumsuz, Karışık, Alakasız
konular: Şu listeden metinle ilgili olanlar (birden fazla olabilir, hiçbiri uymuyorsa boş liste): randevu, fiyat, süre, personel, iş_kalitesi, yedek_parça, temizlik, diğer

Sadece şu biçimde JSON döndür, başka hiçbir şey yazma:
{{"duygu": "...", "konular": ["..."]}}

Geri bildirim: \"\"\"{metin}\"\"\""""

# Prompt 2 (bonus): farklı bir yöntem — önce konu bazlı değerlendirmeleri çıkar, duyguyu bunlardan türet
# Örnekler veride OLMAYAN uydurma cümlelerden; test metinlerine özgü ipucu yok
PROMPT_2 = """Bir otomotiv yetkili servisine gelen müşteri yorumunu analiz edeceksin. Amaç, müşterinin \
neyi nasıl değerlendirdiğini çıkarmak.

Şu adımları izle:

1. Yorumda müşterinin değerlendirdiği her şeyi bul ve her biri için yönünü belirle (olumlu / olumsuz).
   Değerlendirilen şeyi şu gruplardan birine ata:
   randevu, fiyat, süre, personel, iş_kalitesi, yedek_parça, temizlik.
   Hiçbirine uymuyorsa ama somut bir şeyse "diğer" de. Yorumda geçmeyen bir şeyi ekleme.

2. Yönü belirlerken yüzeydeki kelimelere değil, müşterinin gerçekte ne demek istediğine bak.
   İnsanlar alay edebilir ya da olumsuz kalıplarla olumlu bir şey söyleyebilir.

3. Genel duyguyu bu listeden çıkar:
   - Sadece olumlu değerlendirmeler varsa → Olumlu
   - Sadece olumsuz değerlendirmeler varsa → Olumsuz
   - İkisi birden varsa → Karışık
   - Somut bir şey değerlendirilmiyorsa, yorumun genel tonuna bak: memnuniyet → Olumlu, şikâyet → Olumsuz,
     ne memnun ne şikâyetçi (çekingen, kararsız) → Karışık
   - Yorum hizmetle hiç ilgili değilse (anlamsız işaret gibi) → Alakasız

## Örnekler
Yorum: "Ne güzel, aracımı teslim aldığım gün yine arıza lambası yandı."
{{"degerlendirmeler": [{{"konu": "iş_kalitesi", "yon": "olumsuz"}}], "duygu": "Olumsuz", "konular": ["iş_kalitesi"]}}

Yorum: "Fiyat makuldü ama servis alanı oldukça kirliydi."
{{"degerlendirmeler": [{{"konu": "fiyat", "yon": "olumlu"}}, {{"konu": "temizlik", "yon": "olumsuz"}}], "duygu": "Karışık", "konular": ["fiyat", "temizlik"]}}

Yorum: "Aldığım hizmetten pişman değilim."
{{"degerlendirmeler": [], "duygu": "Olumlu", "konular": []}}

Yorum: "Servisin girişinde engelli rampası yok."
{{"degerlendirmeler": [{{"konu": "diğer", "yon": "olumsuz"}}], "duygu": "Olumsuz", "konular": ["diğer"]}}

Sadece bu biçimde JSON döndür, başka hiçbir şey yazma.

Yorum: \"\"\"{metin}\"\"\""""


def llm_etiketle(metin, prompt, deneme=5):
    """Tek bir metni etiketler. JSON zorunlu, temperature=0; geçici hatalarda (yoğunluk, kota) bekleyip tekrar dener."""
    bekleme = 5
    for i in range(deneme):
        try:
            cevap = groq_client.chat.completions.create(
                model=MODEL,
                messages=[{'role': 'user', 'content': prompt.format(metin=metin)}],
                temperature=0,
                response_format={'type': 'json_object'},
            )
            ham = cevap.choices[0].message.content
            return ham, json.loads(ham)
        except Exception as e:
            if i < deneme - 1:
                print(f'  Hata ({type(e).__name__}), {bekleme} sn bekleyip tekrar deneniyor...')
                time.sleep(bekleme)
                bekleme *= 2
            else:
                return f'HATA: {e}', None


def hepsini_etiketle(prompt, dosya_adi):
    """200 yorumun her biri ayrı istek: LLM önceki cevaplarını görmez → aynı metne verdiği cevapların tutarlılığı ölçülebilir.
    Dosya zaten varsa (ve LLM_YENIDEN=False ise) LLM'e gidilmez, kayıtlı sonuç okunur."""
    yol = CIKTI + dosya_adi
    if os.path.exists(yol) and not LLM_YENIDEN:
        print('Kayıtlı sonuç kullanılıyor:', dosya_adi)
        return pd.read_csv(yol, encoding='utf-8-sig').fillna('')
    sonuclar = []
    for i, satir in enumerate(orneklem.itertuples(), start=1):
        ham, sonuc = llm_etiketle(satir.metin, prompt)
        sonuclar.append({
            'geri_bildirim_id': satir.geri_bildirim_id,
            'arac_id': satir.arac_id,
            'metin': satir.metin,
            'duygu': sonuc.get('duygu', '') if sonuc else '',
            'konular': ', '.join(sonuc.get('konular', [])) if sonuc else '',
            'ham_cevap': ham,
        })
        if i % 20 == 0:
            print(f'{i}/200')
            pd.DataFrame(sonuclar).to_csv(yol, index=False, encoding='utf-8-sig')   # ara kayıt
        time.sleep(1)   # ücretsiz katmandaki dakika başına istek sınırı için
    df = pd.DataFrame(sonuclar)
    df.to_csv(yol, index=False, encoding='utf-8-sig')
    print('Bitti:', len(df), 'satır →', dosya_adi)
    return df.fillna('')


# Hızlı test: veride olmayan 2 metin, iki prompt
for metin in ['Usta işini iyi yaptı ama fatura çok kabarık geldi.', 'Teşekkür ederim.']:
    print(metin)
    print('  Prompt 1 →', llm_etiketle(metin, PROMPT_1)[1])
    print('  Prompt 2 →', llm_etiketle(metin, PROMPT_2)[1])

Usta işini iyi yaptı ama fatura çok kabarık geldi.
  Prompt 1 → {'duygu': 'Karışık', 'konular': ['iş_kalitesi', 'fiyat']}
  Prompt 2 → {'degerlendirmeler': [{'konu': 'iş_kalitesi', 'yon': 'olumlu'}, {'konu': 'fiyat', 'yon': 'olumsuz'}], 'duygu': 'Karışık', 'konular': ['iş_kalitesi', 'fiyat']}
Teşekkür ederim.
  Prompt 1 → {'duygu': 'Olumlu', 'konular': []}
  Prompt 2 → {'degerlendirmeler': [], 'duygu': 'Olumlu', 'konular': []}


In [12]:
# Sonuç: İki prompt da geçerli JSON döndürüyor. Prompt 2, duyguya varmadan önce konu bazlı değerlendirmeleri
# ayrı bir alanda gösteriyor → hatalı cevaplarda modelin hangi adımda yanıldığı görülebilecek.

In [21]:
llm_1 = hepsini_etiketle(PROMPT_1, 'llm_etiketleri.csv')

20/200
40/200
60/200
80/200
100/200
120/200
140/200
160/200
180/200
200/200
Bitti: 200 satır → llm_etiketleri.csv


## 3.2 Kalite ölçümü

LLM etiketleri altın etiketlerle metne göre birleştiriliyor. Vakanın istediği 40 kayıtlık altın set, 200 kaydın içinden rastgele seçiliyor. 200 kaydın tamamı da 35 elle etiketlenmiş metinden oluştuğu için, doğruluk ayrıca 200 kayıtta da raporlanıyor.

In [22]:
# Kontrol: LLM cevapları geçerli mi?
print(f"Prompt 1: satır {len(llm_1)} | geçersiz duygu {(~llm_1['duygu'].isin(GECERLI_DUYGU)).sum()} | "
      f"geçersiz konu {llm_1['konular'].apply(gecersiz_konular).str.len().gt(0).sum()}")

def birlestir(llm_df):
    """LLM etiketlerini altın etiketlerle metne göre birleştirir; duygu ve konu eşleşmesini hesaplar."""
    b = llm_df.merge(altin[['metin', 'duygu', 'konular']], on='metin', suffixes=('_llm', '_altin'))
    kume = lambda k: frozenset(x.strip() for x in str(k).split(',') if x.strip())   # sıradan bağımsız konu kümesi
    b['konu_llm'] = b['konular_llm'].apply(kume)
    b['konu_altin'] = b['konular_altin'].apply(kume)
    b['duygu_dogru'] = b['duygu_llm'] == b['duygu_altin']
    b['konu_dogru'] = b['konu_llm'] == b['konu_altin']
    return b

b1 = birlestir(llm_1)
print('Birleşik satır:', len(b1))   # 200 olmalı

# Vakanın istediği 40 kayıtlık altın set: 200 kaydın içinden rastgele
altin_set = b1.sample(n=40, random_state=42)
altin_set[['geri_bildirim_id', 'arac_id', 'metin', 'duygu_altin', 'konular_altin', 'duygu_llm', 'konular_llm']] \
    .to_csv(CIKTI + 'altin_set.csv', index=False, encoding='utf-8-sig')
s40 = b1['geri_bildirim_id'].isin(altin_set['geri_bildirim_id'])

print(f"\nDuygu doğruluğu — 40 kayıt: %{b1.loc[s40, 'duygu_dogru'].mean()*100:.1f} | 200 kayıt: %{b1['duygu_dogru'].mean()*100:.1f}")
print(f"Konu tam eşleşme — 40 kayıt: %{b1.loc[s40, 'konu_dogru'].mean()*100:.1f} | 200 kayıt: %{b1['konu_dogru'].mean()*100:.1f}")

# Hangi duygular birbirine karışıyor? (satır: senin etiketin, sütun: LLM'in etiketi)
print('\nKarışıklık tablosu (200 kayıt):')
print(pd.crosstab(b1['duygu_altin'], b1['duygu_llm'], rownames=['altın'], colnames=['LLM']))

Prompt 1: satır 200 | geçersiz duygu 0 | geçersiz konu 0
Birleşik satır: 200

Duygu doğruluğu — 40 kayıt: %92.5 | 200 kayıt: %89.0
Konu tam eşleşme — 40 kayıt: %62.5 | 200 kayıt: %73.5

Karışıklık tablosu (200 kayıt):
LLM       Alakasız  Karışık  Olumlu  Olumsuz
altın                                       
Alakasız         6        0       0        0
Karışık          0       19       7        6
Olumlu           0        0      72        4
Olumsuz          0        0       5       81


In [23]:
# Sonuç:
# - Duygu doğruluğu: 40 kayıtta %92,5, 200 kayıtta %89 (200'de 22 hata).
# - Konu tam eşleşme: 40 kayıtta %62,5, 200 kayıtta %73,5. Bu sefer 40'lık set 200'den kötü;
#   küçük rastgele örneklem sonucu iki yöne de saptırabiliyor → tek bir 40'lık sayıya güvenmek riskli.
# - Karışıklık tablosu: Hataların çoğu "Karışık"ta: 32 karışık yorumun 13'ü yanlış (7'si Olumlu, 6'sı Olumsuz denmiş).
#   Olumsuz → Olumlu (5) ve Olumlu → Olumsuz (4) hataları da var. Alakasız'da hiç hata yok.

In [24]:
# Amaç: Duygu hataları hangi metinlerde? Her metin için LLM'in bütün cevapları (tutarlılık da görünüyor)
def metin_bazinda_duygu(b):
    t = b.groupby('metin').agg(
        adet=('metin', 'size'),
        altin=('duygu_altin', 'first'),
        llm_cevaplari=('duygu_llm', lambda x: dict(x.value_counts())),
        hatali=('duygu_dogru', lambda x: (~x).sum()),
    )
    return t[t['hatali'] > 0].sort_values('hatali', ascending=False)

pd.set_option('display.max_colwidth', 70)
hatalar_1 = metin_bazinda_duygu(b1)
print(hatalar_1.to_string())
print(f"\n{b1['metin'].nunique()} metnin {len(hatalar_1)}'inde hata var; toplam {hatalar_1['hatali'].sum()} hatalı kayıt")

# Tutarlılık: Aynı metne farklı duygular verilen metin var mı?
tutarsiz = b1.groupby('metin')['duygu_llm'].nunique()
print('Aynı metne birden fazla farklı duygu verilen metin sayısı:', (tutarsiz > 1).sum())

                                                                                adet    altin                 llm_cevaplari  hatali
metin                                                                                                                              
Fena değildi.                                                                      7  Karışık                 {'Olumlu': 7}       7
Harika bir deneyimdi, sadece üç hafta bekledim.                                    5  Olumsuz                 {'Olumlu': 5}       5
Kötü diyemem ama bir daha gelir miyim bilmiyorum.                                  5  Karışık                {'Olumsuz': 5}       5
Hiç de memnun kalmadım denemez, işlerini biliyorlar.                               4   Olumlu                {'Olumsuz': 4}       4
Arıza çözüldü ancak randevu sistemi çok karışık, üç kez aramak zorunda kaldım.     2  Karışık  {'Karışık': 1, 'Olumsuz': 1}       1

35 metnin 5'inde hata var; toplam 22 hatalı kayıt
Aynı metne birden fazla f

In [25]:
# Hata türleri (elle sınıflandırma): hangi farklar gerçek LLM hatası, hangileri görev tanımındaki belirsizlik?
HATA_TURU = {
    'Harika bir deneyimdi, sadece üç hafta bekledim.': 'Gerçek hata: ironi',
    'Hiç de memnun kalmadım denemez, işlerini biliyorlar.': 'Gerçek hata: çift olumsuz',
    'Fena değildi.': 'Belirsiz: çekingen ifade',
    'Kötü diyemem ama bir daha gelir miyim bilmiyorum.': 'Belirsiz: çekingen ifade',
    'Arıza çözüldü ancak randevu sistemi çok karışık, üç kez aramak zorunda kaldım.': 'Tutarsızlık (aynı metne farklı cevap)',
}
hatali = b1[~b1['duygu_dogru']].copy()
hatali['tur'] = hatali['metin'].map(HATA_TURU)
print(hatali['tur'].value_counts().to_string())

# Belirsiz (çekingen) metinler çıkarıldığında doğruluk
belirsiz = b1['metin'].map(HATA_TURU).fillna('').str.startswith('Belirsiz')
print(f"\nDuygu doğruluğu, bütün 200 kayıt: %{b1['duygu_dogru'].mean()*100:.1f}")
print(f"Duygu doğruluğu, çekingen ifadeler hariç ({(~belirsiz).sum()} kayıt): %{b1.loc[~belirsiz, 'duygu_dogru'].mean()*100:.1f}")

# Sonuç:
# - 22 hatanın 9'u gerçek LLM hatası (ironi 5, çift olumsuz 4), 12'si görev tanımındaki belirsizlik
#   (çekingen ifadeler; Prompt 1'de "çekingen → Karışık" kuralı yok), 1'i tutarsızlık.
# - Çekingen ifadeler hariç tutulunca doğruluk %89'dan ~%95'e çıkıyor.
# - Sistematik hata 1 — ironi: olumlu bir kelime ("harika") olumsuz bir olayla birlikte kullanılınca LLM kelimeye takılıyor.
# - Sistematik hata 2 — çift olumsuz: "memnun kalmadım denemez" yapısında olumsuz kısma takılıp anlamın tersine döndüğünü kaçırıyor.
# - temperature=0'a rağmen aynı metne farklı cevap verilebiliyor (1 metin).

tur
Belirsiz: çekingen ifade                 12
Gerçek hata: ironi                        5
Gerçek hata: çift olumsuz                 4
Tutarsızlık (aynı metne farklı cevap)     1

Duygu doğruluğu, bütün 200 kayıt: %89.0
Duygu doğruluğu, çekingen ifadeler hariç (188 kayıt): %94.7


In [26]:
# Amaç: Konu bazında başarı — hangi konuda LLM fazla etiket koyuyor, hangisini kaçırıyor?
def konu_tablosu(b):
    satirlar = []
    for konu in GECERLI_KONU:
        llm_var = b['konu_llm'].apply(lambda s: konu in s)
        altin_var = b['konu_altin'].apply(lambda s: konu in s)
        ortak = (llm_var & altin_var).sum()
        satirlar.append({'konu': konu, 'altin': altin_var.sum(), 'llm': llm_var.sum(), 'ortak': ortak,
                         'kesinlik': ortak / llm_var.sum() if llm_var.sum() else np.nan,
                         'duyarlilik': ortak / altin_var.sum() if altin_var.sum() else np.nan})
    return pd.DataFrame(satirlar).set_index('konu')

print(konu_tablosu(b1).round(2).to_string())

# Konu farkları hangi metinlerde?
def metin_bazinda_konu(b):
    f = b[~b['konu_dogru']]
    return f.groupby('metin').agg(
        adet=('metin', 'size'),
        altin=('konu_altin', lambda x: sorted(x.iloc[0])),
        llm=('konu_llm', lambda x: [sorted(s) for s in x.unique()]),
    ).sort_values('adet', ascending=False)

print()
print(metin_bazinda_konu(b1).to_string())

             altin  llm  ortak  kesinlik  duyarlilik
konu                                                
randevu         15   21     15      0.71        1.00
fiyat           31   31     31      1.00        1.00
süre            47   47     47      1.00        1.00
personel        54   51     41      0.80        0.76
iş_kalitesi     17   35     15      0.43        0.88
yedek_parça     13   13     13      1.00        1.00
temizlik        18   18     18      1.00        1.00
diğer           31   27     25      0.93        0.81

                                                                                   adet                   altin                          llm
metin                                                                                                                                       
İkinci kez geliyorum, yine sorunsuz. Ekibe teşekkürler.                              13              [personel]    [[iş_kalitesi, personel]]
Fatura tutarı verilen tekliften yüksek çıktı, 

In [27]:
# Sonuç:
# - fiyat, süre, yedek_parça, temizlik: hatasız (kesinlik ve duyarlılık 1,00). Açıkça geçen konularda sorun yok.
#   "Yedek araç" yedek parça olarak etiketlenmemiş (yedek_parça kesinliği 1,00).
# - Sistematik hata 3 — iş_kalitesi fazla kullanılıyor (kesinlik 0,43; 35 etiketin 20'si fazladan):
#   LLM "sorunsuz", "mükemmel değildi", "servis iyi" gibi genel ifadelerden iş kalitesi çıkarıyor (metinde olmayanı çıkarma).
# - personel iki yönlü: açıkça anılan "danışman bey"i kaçırıyor (5), çağrı merkezine ulaşamamayı personel sayıyor (6).
# - Farkların bir kısmı belirsiz: "randevu saatinde gittim ama bekledim" (randevu mu?), "bilgilendirilmedim" (personel mi?),
#   "web sitesi" (diğer mi?) — iki taraf da savunulabilir.

## 3.3 Aksiyon önerisi

Müşterilerin terk riski Bölüm 2'deki modelden geliyor (`ciktilar/risk_skorlari.csv`, Bölüm 2'nin son hücresinde üretiliyor); bu yüzden Bölüm 3'ten önce Bölüm 2 çalıştırılmış olmalı. Geri bildirimler müşteriye araç üzerinden bağlanıyor; aracın satın alma tarihinden önceki yorumlar (Bölüm 1: büyük ihtimalle önceki sahibe ait) dışarıda bırakılıyor. Seçimde gerçek terk bilgisi kullanılmıyor.

In [28]:
# Hangi LLM etiketleri kullanılacak? (Prompt 2 tamamlanınca llm_2 yapılacak)
ETIKET = llm_1

risk = pd.read_csv(CIKTI + 'risk_skorlari.csv', index_col='musteri_id')
araclar = pd.read_csv(VERI + 'araclar.csv', parse_dates=['satin_alma_tarihi'])

# Geri bildirim → araç → müşteri (satın alma öncesi yorumlar hariç)
aday = (ETIKET[['geri_bildirim_id', 'arac_id', 'metin', 'duygu', 'konular']]
        .merge(orneklem[['geri_bildirim_id', 'tarih']], on='geri_bildirim_id')
        .merge(araclar[['arac_id', 'musteri_id', 'satin_alma_tarihi']], on='arac_id'))
aday['tarih'] = pd.to_datetime(aday['tarih'])
print('Satın alma öncesi yorum (hariç):', (aday['tarih'] < aday['satin_alma_tarihi']).sum())
aday = aday[aday['tarih'] >= aday['satin_alma_tarihi']]
aday = aday.merge(risk[['risk_skoru']], left_on='musteri_id', right_index=True)   # popülasyon dışı müşteriler düşer
print('Risk skoru olan yorum:', len(aday))

# Seçim: LLM'in "Olumsuz" dediği yorumlar, riske göre sırala, her müşteriden bir kez
olumsuz = aday[aday['duygu'] == 'Olumsuz'].sort_values('risk_skoru', ascending=False)
secilen = olumsuz.drop_duplicates('musteri_id').head(3)
print('\nOlumsuz yorumu olan en riskli müşteriler (ilk 10):')
print(olumsuz.drop_duplicates('musteri_id')[['musteri_id', 'risk_skoru', 'metin', 'konular']].head(10).round(3).to_string(index=False))

# Kontrol: LLM "Olumsuz" demediği için listeye girmeyen yüksek riskli şikâyetçi var mı?
print('\nOlumsuz etiketlenmemiş en riskli yorumlar (ilk 5):')
print(aday[aday['duygu'] != 'Olumsuz'].sort_values('risk_skoru', ascending=False)
      [['musteri_id', 'risk_skoru', 'metin', 'duygu']].head(5).round(3).to_string(index=False))

Satın alma öncesi yorum (hariç): 20
Risk skoru olan yorum: 180

Olumsuz yorumu olan en riskli müşteriler (ilk 10):
 musteri_id  risk_skoru                                                                          metin                 konular
     100796       0.763      Fatura tutarı verilen tekliften yüksek çıktı, önceden bilgilendirilmedim.                   fiyat
     101896       0.759 Arıza çözüldü ancak randevu sistemi çok karışık, üç kez aramak zorunda kaldım.                 randevu
     103816       0.745                    Ödeme sırasında POS çalışmadı, nakit aramak zorunda kaldım.                   diğer
     102259       0.744                                                         Bir yıldız bile fazla.                        
     102827       0.644                  Telefonla kimseye ulaşamıyorum, çağrı merkezi sürekli meşgul.                personel
     102342       0.450          Yedek parça iki hafta bekledi, bu süre boyunca aracım serviste kaldı.       yedek_parça, s

In [29]:
# Sonuç (Prompt 1 etiketleriyle):
# - Satın alma öncesi 20 yorum hariç tutuldu; 180 yorumun müşterisi popülasyonda.
# - Seçilen 3 müşteri: 100796 (fatura yüksek, bilgilendirilmedi), 101896 (arıza çözüldü ama randevu karışık),
#   103816 (POS çalışmadı).
# - 101896, LLM'in tutarsız olduğu cümleden geliyor (aynı metne bir kez Karışık, bir kez Olumsuz) → altın etiketle
#   (Karışık) seçilmezdi. LLM tutarsızlığı, aranacak müşteriyi doğrudan değiştiriyor.
# - KAÇIRILAN: Riski en yüksek şikâyetçi müşteri (103903, risk 0,971) ironik yorumu ("Harika bir deneyimdi, sadece üç hafta
#   bekledim") Olumlu etiketlendiği için listeye girmedi → 3.2'deki ironi hatasının iş açısından bedeli.
# - Riski en yüksek iki müşteri (0,987 / 0,980) memnun ("Sorun yok") → risk modeli ve geri bildirim farklı şeyler ölçüyor.

In [30]:
# Önce: kısıt yok — LLM serbestçe öneri yazıyor (sorunu göstermek için)
PROMPT_SERBEST = """Bir otomotiv servis şirketinde müşteri temsilcisine destek veriyorsun.
Aşağıdaki müşterinin bizi bırakma riski yüksek. Müşteriyi geri kazanmak için
temsilciye yönelik TEK CÜMLELİK bir aksiyon önerisi yaz.

Terk riski: %{risk:.0f}
Müşterinin geri bildirimi: "{metin}"
"""

for _, m in secilen.iterrows():
    cevap = groq_client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': PROMPT_SERBEST.format(risk=m['risk_skoru'] * 100, metin=m['metin'])}],
        temperature=0,
        reasoning_effort=REASONING,
    )
    print(f"Müşteri {m['musteri_id']} (risk %{m['risk_skoru']*100:.0f}): {m['metin']}")
    print(f"  → {cevap.choices[0].message.content.strip()}\n")

Müşteri 100796 (risk %76): Fatura tutarı verilen tekliften yüksek çıktı, önceden bilgilendirilmedim.
  → Müşteriye, teklifin detaylarını ve fatura tutarını önceden net bir şekilde açıklayan, %10 indirimli bir “Ön Bilgilendirme Paketi” sunarak, şeffaflık ve güven oluşturun.

Müşteri 101896 (risk %76): Arıza çözüldü ancak randevu sistemi çok karışık, üç kez aramak zorunda kaldım.
  → Kullanıcı deneyimini iyileştirmek için randevu sistemini tek adımda tamamlanabilir bir akışa dönüştürün ve müşteriye bu yeni basitleştirilmiş süreci telefonla veya SMS ile bildirin.

Müşteri 103816 (risk %75): Ödeme sırasında POS çalışmadı, nakit aramak zorunda kaldım.
  → Müşteriye, ödeme sırasında yaşanan POS arızası için özür dileyip, bir sonraki ziyaretinde ücretsiz bir POS test hizmeti ve nakit ödeme ihtiyacını ortadan kaldıracak bir indirim kuponu sunun.



In [31]:
# Sonuç (kısıtsız prompt): Üç önerinin ikisinde şirketin sunup sunmadığını bilmediğimiz taahhütler uydurulmuş:
# - 100796: "%10 indirimli Ön Bilgilendirme Paketi" → hem uydurma bir ürün hem uydurma bir indirim oranı.
# - 103816: "ücretsiz POS test hizmeti" ve "indirim kuponu" → uydurma hizmet ve kampanya.
# - 101896: "randevu sistemini tek adımda tamamlanabilir hale getirin" → uydurma değil ama temsilcinin
#   yetkisinde olmayan, şirket çapında bir değişiklik; temsilcinin yarın uygulayabileceği bir aksiyon değil.
# Bu öneriler olduğu gibi müşteriye iletilirse şirket ya vermediği bir sözü tutmak zorunda kalır
# ya da müşteri bir kez daha hayal kırıklığına uğrar (verideki "kampanya şubede geçerli değilmiş" şikâyeti gibi).

### Kısıtlı öneri: katalog + yapılandırılmış çıktı + kodla kontrol

Aksiyon kataloğu **varsayımsal**: gerçek uygulamada iş biriminden alınmalı. Bilerek hiçbir parasal taahhüt içermiyor; kampanya gerekiyorsa temsilci şirketin güncel kampanya sisteminden kontrol ediyor (A5).

In [37]:
# Müşteri profili: Bölüm 2 modelinin en önemli değişkenleri (müşterinin NEDEN riskli olduğunu LLM'e anlatmak için)
profil = pd.read_csv(CIKTI + 'analiz_tablosu.csv', index_col='musteri_id')
profil['arac_yasi'] = 2025 - profil['en_yeni_model_yili']

PROMPT_ONERI = """Bir otomotiv yetkili servisinde müşteri temsilcisine destek veriyorsun. Aşağıdaki müşterinin \
servisimizi bırakma riski yüksek. Temsilcinin bu müşteriyi geri kazanmak için yapması gerekeni öner.

## Müşteri
- Terk riski: %{risk:.0f}
- Son servisten bu yana: {recency} gün
- En yeni aracının yaşı: {yas} yıl
- Ortalama memnuniyet puanı (1–5): {memnuniyet}
- Müşteri tipi: {tip}

## Müşterinin geri bildirimi
"{metin}" (konular: {konular})

## Temsilcinin yapabilecekleri
Müşteriyi aramak, özür dilemek, bilgilendirmek, şikâyeti ilgili birime iletip takip etmek, servis müdürüyle görüşme \
ayarlamak, randevu ayarlamak. Şirketin süreçlerini veya sistemlerini değiştiremez.

## Kurallar
- Hiçbir kampanya, indirim, iade, hediye veya ücretsiz hizmet UYDURMA; oran, tutar veya rakam yazma.
- Bir teklif gerekiyorsa, temsilcinin şirket sistemindeki güncel kampanyaları kontrol etmesini söyle.
- Öneri, müşterinin somut şikâyetine ve durumuna (profiline) özel olmalı.

Temsilciye hitap eden, emir kipiyle başlayan TEK CÜMLE yaz.
Sadece JSON döndür: {{"cumle": "..."}}"""

# Kodla kontrol: LLM talimatlara her zaman uymaz → çıktıyı ayrıca denetle
YASAK = re.compile(r'%|\d|indirim|iade|ücretsiz|bedava|hediye|garanti|kupon', re.IGNORECASE)

def kontrol_et(cumle):
    cumle = (cumle or '').strip()
    if not cumle:
        return 'REDDEDİLDİ: boş cevap'
    if YASAK.search(cumle):
        return 'REDDEDİLDİ: yasaklı ifade (kampanya/indirim/rakam)'
    if len(re.findall(r'[.!?](\s|$)', cumle)) > 1:
        return 'REDDEDİLDİ: birden fazla cümle'
    return 'ONAYLANDI'

for _, m in secilen.iterrows():
    p = profil.loc[m['musteri_id']]
    cevap = groq_client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': PROMPT_ONERI.format(
            risk=m['risk_skoru'] * 100, recency=int(p['recency_gun']), yas=int(p['arac_yasi']),
            memnuniyet=f"{p['ort_memnuniyet']:.1f}" if pd.notna(p['ort_memnuniyet']) else 'bilinmiyor',
            tip=p['musteri_tipi'], metin=m['metin'], konular=m['konular'] or 'belirtilmemiş')}],
        temperature=0, reasoning_effort=REASONING, response_format={'type': 'json_object'},
    )
    cumle = json.loads(cevap.choices[0].message.content).get('cumle', '')
    print(f"Müşteri {m['musteri_id']} (risk %{m['risk_skoru']*100:.0f}, {int(p['recency_gun'])} gün, "
          f"araç {int(p['arac_yasi'])} yaş, memnuniyet {p['ort_memnuniyet']:.1f}): {m['metin']}")
    print(f"  Öneri  : {cumle}")
    print(f"  Kontrol: {kontrol_et(cumle)}\n")

Müşteri 100796 (risk %76, 304 gün, araç 10 yaş, memnuniyet 3.7): Fatura tutarı verilen tekliften yüksek çıktı, önceden bilgilendirilmedim.
  Öneri  : Müşteriyi arayarak, fatura tutarının yüksek çıkmasının sebeplerini açıklayarak ve önceden bilgilendirilmediği için özür dileyerek, ilgili birime şikâyeti ilet ve gerekirse servis müdürüyle görüşme ayarlayarak memnuniyetini yeniden kazan.
  Kontrol: ONAYLANDI

Müşteri 101896 (risk %76, 273 gün, araç 12 yaş, memnuniyet 3.8): Arıza çözüldü ancak randevu sistemi çok karışık, üç kez aramak zorunda kaldım.
  Öneri  : Müşteriyi arayarak özür dileyin, randevu sistemindeki karışıklığı anladığınızı belirterek, ilgili birime şikâyeti iletin ve bir sonraki randevuyu hızlıca ayarlayın.
  Kontrol: ONAYLANDI

Müşteri 103816 (risk %75, 289 gün, araç 1 yaş, memnuniyet 3.0): Ödeme sırasında POS çalışmadı, nakit aramak zorunda kaldım.
  Öneri  : Müşteriyi arayarak özür dileyin, POS sorununun çözümü için ilgili birime şikâyeti iletin ve servis müdürüyle görü

In [38]:
# Sonuç: Üç öneri de kontrolden geçti; uydurma kampanya/indirim/ücretsiz hizmet yok, hepsi temsilcinin yetkisindeki aksiyonlar
# (aramak, özür dilemek, açıklamak, ilgili birime iletmek, görüşme/randevu ayarlamak) ve şikâyete özel.
# Kısıtsız prompt'a göre fark: uydurma taahhütler ("%10 indirimli paket", "indirim kuponu") ve temsilcinin yapamayacağı
# öneriler ("randevu sistemini yeniden tasarlayın") ortadan kalktı.
# Sınır: Profil bilgisi verildiği halde öneriler müşterinin 9–10 aydır gelmediğine değinmiyor; hitap tutarsız (sen/siz).

### LLM'in sunulmayan bir kampanyayı önermesini nasıl engellerim?

Kısıtsız prompt üç önerinin ikisinde uydurma taahhüt üretti ("%10 indirimli Ön Bilgilendirme Paketi", "indirim kuponu", "ücretsiz POS test hizmeti"); üçüncüsü temsilcinin yetkisinde olmayan bir sistem değişikliğiydi. Uydurmanın sebebi, LLM'in şirketin hangi kampanyaları sunduğunu bilmemesi ve boşluğu kendi tahminiyle doldurması. Bu yüzden katmanlı bir yaklaşım gerekiyor:

1. **Bilgiyi LLM'in hafızasından değil şirketin sisteminden almak:** LLM'e kampanya önerme yetkisi verilmiyor; teklif gerekiyorsa temsilcinin güncel kampanya sistemini kontrol etmesini söylüyor. Gerçek bir uygulamada aktif kampanyalar sistemden otomatik çekilip prompt'a eklenir (RAG); LLM yalnızca bu listeden önerebilir.
2. **Prompt'ta yetki ve kural tanımlamak:** Temsilcinin yapabilecekleri açıkça listeleniyor; kampanya, indirim, ücretsiz hizmet uydurmak ve rakam yazmak yasaklanıyor. LLM'e müşterinin profili ve şikâyeti veriliyor ki öneri genel değil, müşteriye özel olsun.
3. **Çıktıyı kodla doğrulamak:** Prompt'taki kurallar yeterli değil, LLM talimatlara her zaman uymaz. Çıktı; yüzde, rakam ve "indirim", "ücretsiz", "kupon" gibi kelimeler açısından kodla kontrol ediliyor, tek cümle olup olmadığına bakılıyor.
4. **İnsan onayı:** Öneri müşteriye değil temsilciye gidiyor; temsilci onaylamadan uygulanmıyor.

**Sınırlar:** Kelime kontrolü her durumu yakalayamaz ("yüzde yirmi" gibi yazıyla yazılmış bir oran geçebilir). Bu yüzden 4. katman (insan onayı) vazgeçilmez. Ayrıca bütün aksiyonları sabit bir kataloğa bağlamak da denendi; daha katı bir güvenlik sağlıyor ama katalogun öngörmediği sorunlarda öneriler genelleşiyor. Asıl risk parasal taahhütler olduğu için, yalnızca teklif ve kampanyaları kısıtlayan bu tasarım tercih edildi.